# Fine-tuning QLORA — Qwen2.5-Coder-7B-Instruct on Code4ML

Run All. For each seed (42, 43, 44) the notebook downloads the base model, trains a QLORA adapter on
`dimsmauls/code4ml-annotated-rated` (config `training`), and pushes the adapter to a private Hugging Face repo
under `seed-<seed>/`.

Before running: set the environment variable `HF_TOKEN` to a Hugging Face token with **write** access
(or the login cell will ask for it).

## 1. Install libraries
`transformers` loads the model, `peft` adds the LoRA adapter, `bitsandbytes` provides 8-bit optimizers and
4-bit quantization, `datasets` and `huggingface_hub` read the dataset and push the adapters.
Versions are not pinned; the versions actually used are printed in the next cell for the report.

In [ ]:
%pip install -q -U transformers peft accelerate datasets huggingface_hub bitsandbytes

## 2. Imports and configuration
All values that change between runs live here. Hyperparameters follow the proposal
(Tabel hyperparameter LoRA/QLoRA). Set `PILOT_SAMPLES = 200` and `SEEDS = [42]` for a pilot run.

In [ ]:
import gc, json, os, time
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")  # less fragmentation

import torch
from datasets import load_dataset
from huggingface_hub import HfApi, login
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from transformers import (AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, DataCollatorForSeq2Seq,
                          EarlyStoppingCallback, Trainer, TrainingArguments, set_seed)

# ---- configuration (values from the proposal, Tabel hyperparameter LoRA/QLoRA) ----
METHOD = "qlora"
BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
DATASET = "dimsmauls/code4ml-annotated-rated"
DATASET_CONFIG = "training"
HF_REPO = "dimsmauls/qwen2.5-coder-7b-code4ml-qlora"   # private model repo for the adapters
SEEDS = [42, 43, 44]
PILOT_SAMPLES = None    # e.g. 200 for a quick pilot run; None = full training split

LORA_R, LORA_ALPHA, LORA_DROPOUT = 8, 16, 0.05   # adapter rank, scaling, dropout
# attention (q/k/v/o) and feed-forward (gate/up/down) projections get an adapter
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
LEARNING_RATE = 2e-4
EPOCHS = 3
PER_DEVICE_BATCH = 1   # 4 runs out of memory on 24 GB GPUs for the longest examples
GRAD_ACCUM = 16         # effective batch stays 16, same as the LoRA run
MAX_LENGTH = 2048       # longest example in the training split is ~1,900 tokens
OPTIMIZER = "paged_adamw_8bit"
# bf16 on A100/RTX 30xx+; older GPUs (e.g. Colab T4) fall back to fp16
BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

import peft, transformers
print(torch.cuda.get_device_name(0), "| bf16:", BF16)
print("transformers", transformers.__version__, "| peft", peft.__version__, "| torch", torch.__version__)

## 3. Log in to Hugging Face
Uses `HF_TOKEN` from the environment, otherwise asks for a token. The token needs write access because the
adapters are pushed to `HF_REPO`, which is created as a private repo if it does not exist yet.

In [ ]:
token = os.environ.get("HF_TOKEN")
login(token=token) if token else login()
api = HfApi()
api.create_repo(HF_REPO, private=True, exist_ok=True)
print("pushing adapters to", HF_REPO)

## 4. Load and format the dataset
The training CSV keeps all rows in one file with a `split` column; the `train` and `validation` rows are
used here (`test` is kept for evaluation). Each pair is written in Qwen's chat format
(user = instruction, assistant = code). Instruction tokens get label `-100`, so the loss is computed only on
the code the model must learn to write.

In [ ]:
data = load_dataset(DATASET, DATASET_CONFIG, split="train")
train_rows = data.filter(lambda row: row["split"] == "train")
val_rows = data.filter(lambda row: row["split"] == "validation")
if PILOT_SAMPLES:
    train_rows = train_rows.shuffle(seed=0).select(range(min(PILOT_SAMPLES, len(train_rows))))
    val_rows = val_rows.select(range(min(max(PILOT_SAMPLES // 10, 20), len(val_rows))))
print("train:", len(train_rows), "| validation:", len(val_rows))

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def encode(row):
    """Chat format: user = instruction, assistant = code. Loss only on the code tokens."""
    prompt = [{"role": "user", "content": row["instruction"]}]
    full = prompt + [{"role": "assistant", "content": row["code"]}]
    prompt_text = tokenizer.apply_chat_template(prompt, add_generation_prompt=True, tokenize=False)
    full_text = tokenizer.apply_chat_template(full, tokenize=False)
    # build the text first, then tokenize: works across transformers versions
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(full_text, add_special_tokens=False)["input_ids"][:MAX_LENGTH]
    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
    return {"input_ids": full_ids, "attention_mask": [1] * len(full_ids), "labels": labels[:len(full_ids)]}

columns = train_rows.column_names
train_ds = train_rows.map(encode, remove_columns=columns)
val_ds = val_rows.map(encode, remove_columns=columns)
# pads each batch to its longest example; padded label positions are ignored by the loss
collator = DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100)

## 5. Training function for one seed
Loads a fresh base model (4-bit NF4), attaches the QLORA adapter, trains with early stopping on
validation loss, then saves the adapter with `metrics.json` (best eval loss, minutes, peak VRAM, GPU) and pushes
it to `HF_REPO/seed-<seed>/`. Memory is freed at the end so the next seed starts clean.

In [ ]:
def train_one(seed: int) -> dict:
    set_seed(seed)                              # controls adapter init, data order and dropout
    torch.cuda.reset_peak_memory_stats()        # peak VRAM is reported per seed
    start = time.time()

    quant = BitsAndBytesConfig(load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
                               bnb_4bit_compute_dtype=torch.bfloat16 if BF16 else torch.float16)
    model = AutoModelForCausalLM.from_pretrained(BASE_MODEL, quantization_config=quant, device_map="auto")
    model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)
    # only the adapter weights are trainable; the base model stays frozen
    model = get_peft_model(model, LoraConfig(r=LORA_R, lora_alpha=LORA_ALPHA, lora_dropout=LORA_DROPOUT,
                                             target_modules=TARGET_MODULES, task_type="CAUSAL_LM"))
    model.print_trainable_parameters()
    print(f"batch per GPU {PER_DEVICE_BATCH} x grad accumulation {GRAD_ACCUM} = effective {PER_DEVICE_BATCH * GRAD_ACCUM}")

    out_dir = f"outputs/{METHOD}-seed-{seed}"
    args = TrainingArguments(
        output_dir=out_dir, seed=seed, data_seed=seed,
        num_train_epochs=EPOCHS, learning_rate=LEARNING_RATE,
        per_device_train_batch_size=PER_DEVICE_BATCH, per_device_eval_batch_size=PER_DEVICE_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM, optim=OPTIMIZER,
        bf16=BF16, fp16=not BF16, gradient_checkpointing=True,
        eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
        # keep the epoch with the lowest validation loss; stop if it gets worse
        load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
        logging_steps=10, report_to="none",
    )
    trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
                      data_collator=collator, callbacks=[EarlyStoppingCallback(early_stopping_patience=1)])
    trainer.train()

    adapter_dir = f"{out_dir}/adapter"
    trainer.model.save_pretrained(adapter_dir)  # adapter only (~tens of MB), not the 15 GB base model
    tokenizer.save_pretrained(adapter_dir)
    metrics = {
        "method": METHOD, "seed": seed, "base_model": BASE_MODEL,
        "train_samples": len(train_ds), "validation_samples": len(val_ds),
        "best_eval_loss": trainer.state.best_metric,
        "epochs_run": trainer.state.epoch,
        "train_minutes": round((time.time() - start) / 60, 1),
        "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1024**3, 2),
        "gpu": torch.cuda.get_device_name(0),
    }
    with open(f"{adapter_dir}/metrics.json", "w") as handle:
        json.dump(metrics, handle, indent=2)

    api.upload_folder(repo_id=HF_REPO, folder_path=adapter_dir, path_in_repo=f"seed-{seed}",
                      commit_message=f"{METHOD} adapter seed {seed}")
    print(json.dumps(metrics, indent=2))

    # release GPU memory before the next seed loads a new base model
    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()
    return metrics

## 6. Run all seeds
Trains and pushes seed 42, then 43, then 44. Each finished seed is already on Hugging Face.
If any seed fails, the error is caught (so the shutdown cell below still runs), its traceback is uploaded to
`HF_REPO/errors/`, and the remaining seeds are skipped.

In [ ]:
import traceback
from datetime import datetime, timezone

results, run_error = [], None
try:
    for seed in SEEDS:
        print(f"===== {METHOD} seed {seed} =====")
        results.append(train_one(seed))
        print(f"pushed -> https://huggingface.co/{HF_REPO}/tree/main/seed-{seed}")
except Exception:
    run_error = traceback.format_exc()
    print(run_error)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    try:
        api.upload_file(path_or_fileobj=run_error.encode(), path_in_repo=f"errors/{stamp}.txt",
                        repo_id=HF_REPO, commit_message=f"{METHOD} training error {stamp}")
        print(f"error saved -> https://huggingface.co/{HF_REPO}/tree/main/errors")
    except Exception as upload_exc:  # keep going to the shutdown cell even if the upload fails
        print("could not upload the error log:", upload_exc)

print(json.dumps(results, indent=2))

## 7. Shut the machine down
Stops the machine when **every seed finished and is on Hugging Face**, or when **a seed failed**
(its error is already uploaded to `HF_REPO/errors/`).

- **Colab:** releases the runtime, so compute units stop being used.
- **Vast.ai:** stops this instance with the `vastai` CLI (GPU billing stops; only the small disk fee remains).
  Needs `VAST_API_KEY` (from https://cloud.vast.ai/account/) in the template's environment variables;
  `CONTAINER_ID` is set by Vast.ai. **Without `VAST_API_KEY` the instance cannot stop itself.**

In [ ]:
AUTO_SHUTDOWN = True

def all_seeds_on_hub() -> bool:
    finished = {result["seed"] for result in results}
    files = set(api.list_repo_files(HF_REPO))
    missing = [seed for seed in SEEDS
               if seed not in finished or f"seed-{seed}/adapter_model.safetensors" not in files]
    if missing:
        print("seeds not finished or not on the Hub:", missing)
    return not missing

def shutdown() -> None:
    try:
        from google.colab import runtime  # only importable on Colab
        print("releasing the Colab runtime")
        runtime.unassign()
        return
    except ImportError:
        pass
    import subprocess
    container_id = os.environ.get("CONTAINER_ID")
    api_key = os.environ.get("VAST_API_KEY")
    if not (container_id and api_key):
        print("CONTAINER_ID or VAST_API_KEY not set: STOP THE INSTANCE YOURSELF on cloud.vast.ai")
        return
    subprocess.run(["pip", "install", "-q", "vastai"], check=True)
    print(f"stopping Vast.ai instance {container_id}")
    subprocess.run(["vastai", "stop", "instance", container_id, "--api-key", api_key], check=True)

if not AUTO_SHUTDOWN:
    print("AUTO_SHUTDOWN is off: stop the machine yourself")
elif run_error:
    print("a seed failed (see errors/ on the Hub); shutting down")
    shutdown()
elif all_seeds_on_hub():
    print("all seeds finished and pushed:", SEEDS)
    shutdown()
else:
    print("not all seeds are on the Hub and no error was caught; shutting down anyway to stop billing")
    shutdown()